---
title: "Від чого залежить ціна діаманта: регресійне моделювання"
subtitle: "Лабораторна робота №3. Основи регресійного моделювання"
author: "Демчук Владислав"
date: last-modified
lang: uk
format:
  html:
    toc: true
    toc-depth: 2
    number-sections: true
    code-fold: true
    code-summary: "Код"
    embed-resources: true
    df-print: kable
    fig-dpi: 120
bibliography: references.bib
nocite: |
  @james2023, @nist, @kuhn2022, @sandve2013
execute:
  warning: false
  message: false
jupyter: python3
---

In [ ]:
#| label: setup
#| code-summary: "Пакети й налаштування"
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from IPython.display import Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
KEY = {}


def save_figure(name):
    plt.savefig("figures/" + name + ".png", dpi=150, bbox_inches="tight")


def rnd(x, digits=0):
    if digits == 0:
        return int(round(float(x)))
    return round(float(x), digits)


def usd(value):
    return f"{value:,.0f}".replace(",", " ") + " $"

# Вступ

## Предметна область, ціль і питання

Ціна діаманта на роздрібному ринку визначається так званими «4C»: вагою (*carat*), огранкою (*cut*),
кольором (*color*) і чистотою (*clarity*). Продовжую лабораторну №2 на тому самому наборі **Diamonds**.

- **Одиниця спостереження** — один огранений круглий діамант з каталогу онлайн-продавця.
- **Ціль** — `price`, ціна в доларах США (ціни 2008 року), числова й неперервна.
- **Дослідницьке питання.** Наскільки точно чотири характеристики з сертифіката (4C) прогнозують ціну
  *нового* діаманта з того самого ринку і як ціна пов'язана з вагою, якщо огранка, колір і чистота однакові?
- **Для яких спостережень потрібен прогноз.** Для круглих діамантів того самого типу й періоду,
  у межах діапазону даних (0.2–5 ct, ціна до 18 823 $). Наприклад, покупець чи продавець хоче оцінити,
  чи справедлива ціна каменя з відомими 4C.

## Що підказує література

1. **Chu [-@chu2001]** будує модель ціни діамантів за 4C на даних сінгапурських оголошень. Ціна зростає
   з вагою швидше, ніж пропорційно, а якісні характеристики працюють як надбавки до ціни.
   Звідси **вибір змінних** (4C як основні предиктори) і **форма моделі**: логарифм ціни дає
   мультиплікативні (відсоткові) ефекти колір/чистота і рівномірніший розкид залишків.
2. **Scott і Yelowitz [-@scott2010]** на даних трьох великих онлайн-продавців показали **стрибки ціни на
   «круглих» вагах**: камінь 0.5 ct дорожчий за 0.49 ct на 18 % і більше, а 1 ct дорожчий за трохи легший
   на 5–10 %. Це підказує **межі інтерпретації**: зв'язок ціни з вагою не гладкий, тому проста
   степенева модель систематично помилятиметься біля порогів 0.5, 1, 1.5, 2 ct. Звідси моя модифікація
   регресії (порогові змінні і сплайн).
3. **Wickham і Grolemund [-@r4ds]** на цьому самому наборі показують, що в шкалі log–log зв'язок ціни з
   вагою майже лінійний і що «гірша» якість здається дорожчою через змішування з вагою. Це методологічна
   підстава завжди оцінювати ефекти якості **за сталої ваги**.

## Гіпотези (записані до оцінювання моделей)

- **H1 (вага, з EDA і літератури).** За однакових огранки, кольору й чистоти ціна зростає з вагою
  **швидше, ніж пропорційно**: еластичність ціни за вагою $\beta_{\log carat} > 1$.
  *Перевірка:* знак, величина й 95 % довірчий інтервал коефіцієнта при $\log(carat)$ у заздалегідь
  описаній моделі M2 (нижче), стійкі до гетероскедастичності стандартні похибки HC3.
- **H2 (чистота, з EDA і літератури).** За однакової ваги, огранки й кольору ціна **монотонно зростає**
  з чистотою від I1 до IF, і надбавка IF порівняно з I1 перевищує 100 %.
  *Перевірка:* коефіцієнти рівнів `clarity` у M2: знаки, порядок і 95 % ДІ надбавки $e^{\beta}-1$.

::: {.callout-warning}
Обидві гіпотези **виникли з EDA** лабораторної №2 (нахил 1.68 у log–log; чистота найсильніше
пов'язана з ціною каменів 1 ct), яке охоплювало **весь** набір. Тому їх перевірка на тих самих даних
є **розвідувальною**, а не підтверджувальною.
:::

# Методи

## Дані

In [ ]:
#| label: data
RAW = "data/raw/diamonds.csv"
URL = "https://raw.githubusercontent.com/tidyverse/ggplot2/main/data-raw/diamonds.csv"
if not os.path.exists(RAW):
    os.makedirs("data/raw", exist_ok=True)
    pd.read_csv(URL).to_csv(RAW, index=False)

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
raw = pd.read_csv(RAW)

df = raw.drop_duplicates()
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]
depth_from_size = df["z"] / ((df["x"] + df["y"]) / 2) * 100
df = df[(depth_from_size - df["depth"]).abs() <= 5]
df = df.reset_index(drop=True)

cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
df["cut"] = pd.Categorical(df["cut"], categories=cut_order, ordered=True)
df["color"] = pd.Categorical(df["color"], categories=color_order, ordered=True)
df["clarity"] = pd.Categorical(df["clarity"], categories=clarity_order, ordered=True)

df["log_price"] = np.log(df["price"])
df["log_carat"] = np.log(df["carat"])

KEY["n_raw"] = len(raw)
KEY["n_clean"] = len(df)
KEY["sha"] = sha
print("Рядків у файлі:", len(raw))
print("Рядків після очищення:", len(df))
print("SHA-256 файлу:", sha)

- **Джерело:** набір `diamonds` з пакета ggplot2 [@ggplot2], файл `data-raw/diamonds.csv` з репозиторію
  tidyverse/ggplot2 (гілка `main`), отримано **листопад 2026**. Контрольна сума SHA-256 файлу
  `{python} sha[:16]`… записана в звіті, тож ту саму версію можна перевірити.
- **Умови використання:** ліцензія MIT пакета ggplot2; дані відкриті.
- **Розмір:** `{python} f"{len(raw):,}".replace(",", " ")` рядків і 10 змінних; після очищення
  `{python} f"{len(df):,}".replace(",", " ")` (вилучено повні дублікати, камені з нульовими розмірами і
  рядки, де `depth` суперечить розмірам x, y, z).
- **Обмеження даних:** спостережні дані одного ринку 2008 року; **немає дати й продавця**; ціни обрізані
  зверху (максимум 18 823 $); у наборі тільки круглі камені. Модель не можна переносити на інші роки,
  форми чи синтетичні діаманти.

## Поділ і захист від витоку

Спостереження незалежні (один рядок — окремий камінь, повторних вимірювань і часу немає), тому
**випадковий поділ 80/20 з `seed = 42`**. Поділ я зафіксував **до** підбору моделей; тестову частину
не використовую аж до фінальної оцінки.

In [ ]:
#| label: split
train, test = train_test_split(df, test_size=0.2, random_state=SEED)

split_table = pd.DataFrame({
    "частина": ["навчальна", "тестова"],
    "рядків": [len(train), len(test)],
    "середня ціна, $": [rnd(train["price"].mean()), rnd(test["price"].mean())],
})
split_table

- Усі графіки розділу 2.3 і всю діагностику я побудував **тільки на навчальній частині**.
- **Обмеження незалежності тесту.** Лабораторна №2 бачила весь набір, зокрема майбутню тестову частину.
  Вона вже підказала рішення: логарифмувати ціну й вагу, відкинути x, y, z (дублюють вагу), не брати
  `depth` і `table` (слабкий зв'язок), а також гіпотези H1–H2. Тому тестова оцінка трохи оптимістична.
  Нових даних з того самого ринку немає, отже повністю незалежної перевірки зробити не можу.
- **Доступність предикторів.** 4C вказані в сертифікаті до продажу, тож вони відомі на момент прогнозу.
  Ціну я ніде не використовую як предиктор.
- **Перехресна валідація:** 5-кратна `KFold(shuffle=True, random_state=42)` на навчальній частині,
  **однакові розбиття для всіх моделей**. Кожну модель на кожному розбитті я навчаю тільки на його
  навчальній частині, а помилку рахую на валідаційній. Лінійні моделі задані формулами `statsmodels`:
  кодування категорій і вузли сплайна визначаються лише за навчальною частиною розбиття, а логарифми й
  пороги ваги рахуються для кожного рядка окремо і нічого не беруть з інших даних.
  Пропусків немає, тож імпутація не потрібна; масштабування лінійній регресії без штрафу не потрібне.

## Що з EDA переношу в моделювання

In [ ]:
#| label: fig-eda
#| fig-cap: "EDA на навчальній частині. Ліворуч: ціна має сильну праву асиметрію, після логарифма розподіл майже симетричний (кілька піків — популярні ваги каменів). Праворуч: у шкалі log–log ціна майже лінійно залежить від ваги, але видно вертикальні «сходинки» на 0.5, 1, 1.5 і 2 ct."
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

sns.histplot(train["price"], bins=60, ax=ax[0], color="steelblue")
ax[0].set(title="Ціна", xlabel="Ціна, $", ylabel="Кількість діамантів")

sns.histplot(train["log_price"], bins=60, ax=ax[1], color="seagreen")
ax[1].set(title="Логарифм ціни", xlabel="ln(ціна, $)", ylabel="")

ax[2].scatter(train["carat"], train["price"], s=2, alpha=0.08)
ax[2].set(xscale="log", yscale="log", title="Ціна і вага (log–log)",
          xlabel="Вага, ct (log)", ylabel="Ціна, $ (log)")
for weight in [0.5, 1, 1.5, 2]:
    ax[2].axvline(weight, color="red", lw=0.8, ls="--")
ax[2].minorticks_off()
ax[2].set_xticks([0.2, 0.3, 0.5, 1, 2, 3, 5], ["0.2", "0.3", "0.5", "1", "2", "3", "5"])
ax[2].set_yticks([500, 1000, 2000, 5000, 10000], ["500", "1000", "2000", "5000", "10000"])

plt.tight_layout()
save_figure("01_eda_target")
plt.show()

KEY["skew_price"] = rnd(train["price"].skew(), 2)
KEY["skew_log"] = rnd(train["log_price"].skew(), 2)

In [ ]:
#| label: fig-eda-clarity
#| fig-cap: "Медіанна ціна за чистотою: без контролю ваги (ліворуч) найчистіші камені здаються дешевшими; серед каменів 0.9–1.1 ct (праворуч) ціна чітко зростає з чистотою. Отже, ефекти якості треба оцінювати за сталої ваги."
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

median_all = train.groupby("clarity", observed=True)["price"].median()
one_carat = train[(train["carat"] >= 0.9) & (train["carat"] <= 1.1)]
median_1ct = one_carat.groupby("clarity", observed=True)["price"].median()

ax[0].bar(median_all.index.astype(str), median_all.values, color="lightgray")
ax[0].set(title="Усі камені", xlabel="Чистота (від гіршої до кращої)", ylabel="Медіанна ціна, $")
ax[1].bar(median_1ct.index.astype(str), median_1ct.values, color="steelblue")
ax[1].set(title="Камені 0.9–1.1 ct", xlabel="Чистота (від гіршої до кращої)", ylabel="")

plt.tight_layout()
save_figure("02_eda_clarity")
plt.show()

Найважливіші висновки з лабораторної №2, перевірені тут на навчальній частині:

- **Розподіл цілі:** асиметрія ціни `{python} KEY["skew_price"]`, логарифма ціни `{python} KEY["skew_log"]`.
  *Рішення:* модель для $\ln(price)$; ефекти тоді у відсотках, а розкид залишків рівніший.
- **Пропуски:** немає. **Викиди:** дорогі й великі камені реальні, їх не видаляю; помилкові рядки
  вже вилучено на очищенні.
- **Зв'язки й нелінійність:** зв'язок ціни з вагою степеневий (у log–log майже пряма) зі «сходинками»
  на круглих вагах. *Рішення:* $\ln(carat)$ у базовій лінійній моделі; сплайн і пороги в модифікації.
- **Змішування:** «краща» якість частіше в дрібних каменях (@fig-eda-clarity). *Рішення:* завжди
  тримаю вагу в моделі, коли інтерпретую огранку, колір і чистоту.
- **Дублювання:** x, y, z майже ідеально корелюють з вагою (r ≈ 0.98), тому не входять у модель.

## Метрики й правило вибору (зафіксовано до відкриття тесту)

- **Основна метрика — MAE**, у доларах: «на скільки в середньому помиляється оцінка ціни». Розподіл ціни
  дуже асиметричний, і RMSE домінували б кілька найдорожчих каменів. RMSE і $R^2$ подаю як додаткові.
- **Правило вибору:** модель з найменшим середнім MAE на 5 розбиттях. Якщо простіша модель гірша менше
  ніж на одне стандартне відхилення MAE за розбиттями, беру простішу.
- На тесті один раз оцінюю обрану модель і константний орієнтир. Найкращу **лінійну** модель (вона
  потрібна для інтерпретації гіпотез) теж один раз оцінюю на тесті, це рішення прийняте заздалегідь.
- Для моделей на $\ln(price)$ прогноз у доларах — $e^{\hat y}$, тобто оцінка **медіани** ціни.
  Вона узгоджується з MAE, але трохи занижує середню ціну [@duan1983].

# Результати

In [ ]:
#| label: models
#| code-summary: "Усі моделі: формули і функція прогнозу"
QUALITY = "C(cut) + C(color) + C(clarity)"
SPLINE = "bs(log_carat, df=6, lower_bound=-2.5, upper_bound=2)"
STEPS = "I(carat >= 0.5) + I(carat >= 1) + I(carat >= 1.5) + I(carat >= 2)"

FORMULAS = {
    "M1": "price ~ carat + " + QUALITY,
    "M2": "log_price ~ log_carat + " + QUALITY,
    "M3": "log_price ~ " + SPLINE + " + " + STEPS + " + " + QUALITY,
    "M4": "log_price ~ " + SPLINE + " + " + STEPS + " + C(cut) + C(color) * C(clarity)",
}

MODEL_NAMES = {
    "M0": "M0 константа (середнє)",
    "M1": "M1 лінійна, ціна в $",
    "M2": "M2 лінійна log–log",
    "M3": "M3 log–log + сплайн і пороги ваги",
    "M4": "M4 = M3 + взаємодія колір×чистота",
    "M5": "M5 градієнтний бустинг (додатково)",
}


def boosting_features(data):
    return pd.DataFrame({
        "carat": data["carat"],
        "cut": data["cut"].cat.codes,
        "color": data["color"].cat.codes,
        "clarity": data["clarity"].cat.codes,
    })


def predict_price(code, fit_data, new_data):
    if code == "M0":
        return np.full(len(new_data), fit_data["price"].mean())

    if code == "M5":
        booster = HistGradientBoostingRegressor(random_state=SEED)
        booster.fit(boosting_features(fit_data), fit_data["log_price"])
        return np.exp(booster.predict(boosting_features(new_data)))

    formula = FORMULAS[code]
    model = smf.ols(formula, data=fit_data).fit()
    prediction = np.asarray(model.predict(new_data))
    if formula.startswith("log_price"):
        prediction = np.exp(prediction)
    return prediction

## Орієнтир: константний прогноз

M0 завжди прогнозує середню ціну навчальної частини, `{python} usd(train["price"].mean())`. Це точність
«без жодних предикторів»: будь-яка корисна модель має бути помітно кращою (результати в @tbl-cv).

## Базова регресія

Спершу найпростіша лінійна модель у доларах (M1):
$price = \beta_0 + \beta_1\,carat + \text{cut} + \text{color} + \text{clarity} + \varepsilon$.

In [ ]:
#| label: fig-m1-resid
#| fig-cap: "M1 (ціна в $): залишки проти прогнозів. Видно дугу (нелінійність), «лійку» (розкид зростає з ціною) і від'ємні прогнози для дрібних каменів."
m1 = smf.ols(FORMULAS["M1"], data=train).fit()

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.scatter(m1.fittedvalues, m1.resid, s=2, alpha=0.1)
ax.axhline(0, color="red")
ax.set(xlabel="Прогноз, $", ylabel="Залишок, $", title="M1: залишки проти прогнозів")
plt.tight_layout()
save_figure("03_m1_residuals")
plt.show()

KEY["m1_neg"] = int((m1.fittedvalues < 0).sum())

M1 має три вади: `{python} KEY["m1_neg"]` каменів отримують **від'ємну** ціну, залишки утворюють дугу
(зв'язок не лінійний), а розкид росте з ціною. Тому як **базову регресію для висновків** беру
лог-лінійну модель, яку підказали EDA й література (M2):

$$\ln(price) = \beta_0 + \beta_1 \ln(carat) + \sum \gamma_{cut} + \sum \delta_{color} + \sum \theta_{clarity} + \varepsilon$$

Еталонні рівні: огранка Fair, колір J, чистота I1. Коефіцієнти я оцінив на всій навчальній частині
OLS зі стійкими до гетероскедастичності похибками HC3.

In [ ]:
#| label: tbl-m2
#| tbl-cap: "M2: ключові коефіцієнти, 95 % ДІ (HC3) і надбавка до ціни за сталих інших змінних"
m2 = smf.ols(FORMULAS["M2"], data=train).fit(cov_type="HC3")
ci = m2.conf_int()

coef = pd.DataFrame({
    "β": m2.params,
    "ДІ 2.5 %": ci[0],
    "ДІ 97.5 %": ci[1],
    "надбавка, %": (np.exp(m2.params) - 1) * 100,
})
coef.loc[["Intercept", "log_carat"], "надбавка, %"] = np.nan
coef = coef.round(3)
coef.to_csv("results/m2_coefficients.csv")

b1 = m2.params["log_carat"]
clarity_terms = ["C(clarity)[T." + level + "]" for level in clarity_order[1:]]
clarity_beta = m2.params[clarity_terms]

KEY["b1"] = rnd(b1, 3)
KEY["b1_lo"] = rnd(ci.loc["log_carat", 0], 3)
KEY["b1_hi"] = rnd(ci.loc["log_carat", 1], 3)
KEY["r2_m2"] = rnd(m2.rsquared, 4)
KEY["si2_prem"] = rnd((np.exp(m2.params["C(clarity)[T.SI2]"]) - 1) * 100)
KEY["if_prem"] = rnd((np.exp(m2.params["C(clarity)[T.IF]"]) - 1) * 100)
KEY["if_lo"] = rnd((np.exp(ci.loc["C(clarity)[T.IF]", 0]) - 1) * 100)
KEY["if_hi"] = rnd((np.exp(ci.loc["C(clarity)[T.IF]", 1]) - 1) * 100)
KEY["clar_monotone"] = bool(clarity_beta.is_monotonic_increasing and (clarity_beta > 0).all())
KEY["d_prem"] = rnd((np.exp(m2.params["C(color)[T.D]"]) - 1) * 100)
KEY["ideal_prem"] = rnd((np.exp(m2.params["C(cut)[T.Ideal]"]) - 1) * 100)

coef

**Інтерпретація в одиницях задачі** ($R^2$ на шкалі логарифма = `{python} KEY["r2_m2"]`):

- **Вага:** $\hat\beta_1 =$ `{python} KEY["b1"]` (95 % ДІ `{python} KEY["b1_lo"]`–`{python} KEY["b1_hi"]`).
  Камінь на 1 % важчий коштує в середньому на ≈ `{python} rnd(b1, 2)` % дорожче. Камінь, удвічі
  важчий за ту саму якість, коштує в $2^{\hat\beta_1} \approx$ `{python} rnd(2 ** b1, 1)` раза більше.
- **Чистота:** порівняно з I1 надбавка зростає від `{python} KEY["si2_prem"]` % (SI2) до
  `{python} KEY["if_prem"]` % (IF, 95 % ДІ `{python} KEY["if_lo"]`–`{python} KEY["if_hi"]` %).
- **Колір:** найкращий колір D дорожчий за J на ≈ `{python} KEY["d_prem"]` %.
- **Огранка:** Ideal дорожча за Fair лише на ≈ `{python} KEY["ideal_prem"]` %. Огранка важить найменше з 4C.

### Діагностика залишків M2

In [ ]:
#| label: fig-m2-diag
#| fig-cap: "Діагностика M2. (а) Залишки проти прогнозів: дуги й лійки M1 майже немає. (б) Середній залишок за вагою: систематична хвиля від −0.12 до +0.11 зі зламами біля 0.5, 1, 1.5 і 2 ct. (в) Q–Q графік: правий хвіст важчий за нормальний."
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

ax[0].scatter(m2.fittedvalues, m2.resid, s=2, alpha=0.08)
ax[0].axhline(0, color="red")
ax[0].set(xlabel="Прогноз ln(ціна)", ylabel="Залишок", title="(а) Залишки проти прогнозів")

weight_bins = np.round(np.arange(0.2, 2.31, 0.05), 2)
weight_group = pd.cut(train["carat"], weight_bins)
resid_by_weight = m2.resid.groupby(weight_group, observed=True).agg(["mean", "size"])
resid_by_weight = resid_by_weight[resid_by_weight["size"] >= 30]
bin_middle = [interval.mid for interval in resid_by_weight.index]
ax[1].plot(bin_middle, resid_by_weight["mean"], marker="o", ms=3)
ax[1].axhline(0, color="red")
for weight in [0.5, 1, 1.5, 2]:
    ax[1].axvline(weight, color="gray", ls="--", lw=0.8)
ax[1].set(xlabel="Вага, ct", ylabel="Середній залишок (ln)", title="(б) Середній залишок за вагою")

stats.probplot(m2.resid.sample(4000, random_state=SEED), dist="norm", plot=ax[2])
ax[2].set_title("(в) Q–Q графік залишків (4000 точок)")

plt.tight_layout()
save_figure("04_m2_diagnostics")
plt.show()

In [ ]:
#| label: influence
m2_plain = smf.ols(FORMULAS["M2"], data=train).fit()

bp_m1 = het_breuschpagan(m1.resid, m1.model.exog)[1]
bp_m2 = het_breuschpagan(m2_plain.resid, m2_plain.model.exog)[1]

cook = OLSInfluence(m2_plain).cooks_distance[0]
n_cook = int((cook > 4 / len(train)).sum())

n_top = int(0.01 * len(train))
top_rows = np.argsort(cook)[-n_top:]
train_without_top = train.drop(train.index[top_rows])
m2_without_top = smf.ols(FORMULAS["M2"], data=train_without_top).fit()

spread_expensive = m1.resid[m1.fittedvalues > 10000].std()
spread_cheap = m1.resid[m1.fittedvalues < 2000].std()

KEY["n_cook"] = n_cook
KEY["cook_max"] = rnd(cook.max(), 4)
KEY["b1_wo"] = rnd(m2_without_top.params["log_carat"], 3)
KEY["sd_ratio"] = rnd(spread_expensive / spread_cheap, 1)

pd.DataFrame({
    "перевірка": ["Breusch–Pagan, p (M1)", "Breusch–Pagan, p (M2)",
                  "Кук D > 4/n, кількість", "Максимальна D Кука",
                  "β ln(carat) без 1 % найвпливовіших"],
    "значення": [f"{bp_m1:.1e}", f"{bp_m2:.1e}", n_cook, KEY["cook_max"], KEY["b1_wo"]],
})

**Що показала діагностика і що з цього випливає:**

- **Нелінійність.** Логарифми прибрали велику дугу M1, але лишилась систематична «хвиля» за вагою
  (@fig-m2-diag б). Камені 0.4–0.5 ct і 0.55–0.85 ct модель переоцінює (середній залишок до −0.12,
  тобто прогноз на ≈ 11 % завищений), а камені близько 1 ct і найдрібніші недооцінює (до +0.11). Злами
  кривої припадають на 0.5, 1, 1.5 і 2 ct, тобто на «круглі» ваги [@scott2010]. *Наслідок для прогнозу:*
  систематична похибка до ≈ 10 % для цілих груп каменів. *Наслідок для коефіцієнтів:* $\beta_1$ є усередненою
  еластичністю, а не сталою для всіх ваг.
- **Неоднаковий розкид.** У M1 розкид залишків для дорогих каменів у `{python} KEY["sd_ratio"]` раза
  більший, ніж для дешевих. У M2 він значно рівніший, проте тест Бройша–Пагана все одно відхиляє
  гомоскедастичність (при n ≈ 43 000 він чутливий навіть до малих відхилень). Тому для ДІ я використав
  **HC3**. Q–Q графік показує важкі хвости, тож прогнозні інтервали на основі нормальності були б
  завузькими.
- **Впливові спостереження.** `{python} KEY["n_cook"]` точок мають D Кука > 4/n, але максимум лише
  `{python} KEY["cook_max"]` (далеко від 1). Без 1 % найвпливовіших точок
  $\hat\beta_1$ = `{python} KEY["b1_wo"]` замість `{python} KEY["b1"]`. Висновки від них не залежать.

## Спроба покращення

**Проблема:** хвиля в залишках за вагою (@fig-m2-diag б). **Модифікації:**

- **M3** замінює пряму $\ln(carat)$ на кубічний сплайн від $\ln(carat)$ (6 ступенів свободи, вузли
  за навчальною частиною) і додає 4 порогові змінні $[carat \ge 0.5], [\ge 1], [\ge 1.5], [\ge 2]$
  з роботи Scott і Yelowitz.
- **M4** додатково дозволяє надбавці за колір залежати від чистоти (взаємодія color × clarity).
  У лабораторній №2 теплова карта медіанних цін «колір × чистота» для каменів 1 ct показала, що ці
  надбавки не просто додаються одна до одної.

Ridge не розглядаю: предикторів лише ~20 на ~43 000 спостережень, мультиколінеарності між 4C немає,
тож штраф майже не змінить коефіцієнтів.

## Додаткова модель

**M5** — градієнтний бустинг (`HistGradientBoostingRegressor`, параметри за замовчуванням) на тих самих
4 предикторах і тій самій цілі $\ln(price)$. Він сам знаходить пороги й взаємодії, але не має
коефіцієнтів для інтерпретації.

## Порівняння за перехресною валідацією

In [ ]:
#| label: tbl-cv
#| tbl-cap: "5-кратна перехресна валідація на навчальній частині (однакові розбиття). Метрики в доларах; середнє ± стандартне відхилення за 5 розбиттями."
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for code in MODEL_NAMES:
    mae_list = []
    rmse_list = []
    r2_list = []
    for fit_index, valid_index in kfold.split(train):
        fit_part = train.iloc[fit_index]
        valid_part = train.iloc[valid_index]
        prediction = predict_price(code, fit_part, valid_part)
        mae_list.append(mean_absolute_error(valid_part["price"], prediction))
        rmse_list.append(root_mean_squared_error(valid_part["price"], prediction))
        r2_list.append(r2_score(valid_part["price"], prediction))
    cv_rows.append({
        "code": code,
        "модель": MODEL_NAMES[code],
        "mae": np.mean(mae_list),
        "mae_sd": np.std(mae_list),
        "rmse": np.mean(rmse_list),
        "rmse_sd": np.std(rmse_list),
        "r2": np.mean(r2_list),
        "r2_sd": np.std(r2_list),
    })

cv = pd.DataFrame(cv_rows).set_index("code")

cv_table = pd.DataFrame({
    "модель": cv["модель"],
    "MAE, $": cv["mae"].map("{:.0f}".format) + " ± " + cv["mae_sd"].map("{:.0f}".format),
    "RMSE, $": cv["rmse"].map("{:.0f}".format) + " ± " + cv["rmse_sd"].map("{:.0f}".format),
    "R²": cv["r2"].map("{:.3f}".format) + " ± " + cv["r2_sd"].map("{:.3f}".format),
})
cv_table.to_csv("results/cv_comparison.csv", index=False)
cv_table.reset_index(drop=True)

In [ ]:
#| label: choice
best = cv["mae"].idxmin()
chosen = best
for code in ["M1", "M2", "M3", "M4", "M5"]:
    if code == best:
        break
    if cv.loc[code, "mae"] - cv.loc[best, "mae"] < cv.loc[best, "mae_sd"]:
        chosen = code
        break

best_linear = cv.loc[["M1", "M2", "M3", "M4"], "mae"].idxmin()

KEY["chosen"] = MODEL_NAMES[chosen]
KEY["best_linear"] = MODEL_NAMES[best_linear]
KEY["cv"] = cv[["модель", "mae", "mae_sd", "rmse"]].round(1).to_dict("records")
KEY["gain_m3"] = rnd((1 - cv.loc["M3", "mae"] / cv.loc["M2", "mae"]) * 100, 1)
KEY["gain_m4"] = rnd((1 - cv.loc["M4", "mae"] / cv.loc["M3", "mae"]) * 100, 1)
KEY["gain_gb"] = rnd((1 - cv.loc["M5", "mae"] / cv.loc[best_linear, "mae"]) * 100, 1)

Markdown("**Обрана модель (до відкриття тесту): " + MODEL_NAMES[chosen] + ".** "
         + "Найкраща лінійна модель: " + MODEL_NAMES[best_linear] + ".")

**Що показала валідація:**

- Усі регресії значно кращі за орієнтир M0. Перехід до логарифмів (M1 → M2) зменшує MAE приблизно
  вдвічі. Отже, форма зв'язку важить більше за вибір предикторів.
- **Сплайн і пороги (M3)** зменшують MAE ще на `{python} KEY["gain_m3"]` % порівняно з M2, різниця
  набагато більша за розкид між розбиттями. Модифікація розв'язала проблему, яку показала діагностика.
- **Взаємодія колір × чистота (M4)** зменшує MAE ще на `{python} KEY["gain_m4"]` % порівняно з M3.
- **Бустинг (M5)** точніший за найкращу лінійну модель на `{python} KEY["gain_gb"]` %, але не має
  коефіцієнтів. Тому гіпотези перевіряю в M2, а для прогнозу беру модель за правилом вибору.

## Фінальна оцінка на тесті

In [ ]:
#| label: tbl-test
#| tbl-cap: "Одноразова оцінка на тестовій частині. Моделі перенавчено на всій навчальній частині."
test_models = ["M0", chosen]
if best_linear != chosen:
    test_models.append(best_linear)

test_predictions = {}
test_rows = []
for code in test_models:
    prediction = predict_price(code, train, test)
    test_predictions[code] = prediction
    relative_error = np.abs(test["price"] - prediction) / test["price"] * 100
    test_rows.append({
        "модель": MODEL_NAMES[code],
        "MAE, $": rnd(mean_absolute_error(test["price"], prediction)),
        "RMSE, $": rnd(root_mean_squared_error(test["price"], prediction)),
        "R²": rnd(r2_score(test["price"], prediction), 3),
        "медіана |помилки| / ціна, %": rnd(np.median(relative_error), 1),
    })

test_table = pd.DataFrame(test_rows)
test_table.to_csv("results/test_metrics.csv", index=False)
KEY["test"] = test_table.to_dict("records")
test_table

In [ ]:
#| label: fig-test
#| fig-cap: "Тест: фактична ціна проти прогнозу обраної моделі (log-шкали; червона лінія — ідеальний прогноз) і абсолютна помилка за групами ваги."
err = test.copy()
err["pred"] = test_predictions[chosen]
err["abs_err"] = np.abs(err["price"] - err["pred"])
err["rel_err"] = err["abs_err"] / err["price"] * 100
err["група ваги"] = pd.cut(err["carat"], [0, 0.5, 0.75, 1, 1.5, 2, 5.1],
                           labels=["до 0.5", "0.5–0.75", "0.75–1", "1–1.5", "1.5–2", "понад 2"])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].scatter(err["price"], err["pred"], s=3, alpha=0.15)
ax[0].plot([300, 20000], [300, 20000], color="red")
ax[0].set(xscale="log", yscale="log", xlim=[300, 20000], ylim=[300, 20000],
          xlabel="Фактична ціна, $", ylabel="Прогноз, $", title="Фактичне і прогноз: " + chosen)
sns.boxplot(data=err, x="група ваги", y="abs_err", ax=ax[1], showfliers=False, color="lightblue")
ax[1].set(xlabel="Вага, ct", ylabel="|помилка|, $", title="Абсолютна помилка за вагою")
plt.tight_layout()
save_figure("05_test_pred")
plt.show()

by_weight = err.groupby("група ваги", observed=True).agg(
    n=("price", "size"),
    median_price=("price", "median"),
    MAE=("abs_err", "mean"),
    median_rel_err=("rel_err", "median"),
).round(1)
by_weight.to_csv("results/test_errors_by_carat.csv")
KEY["by_carat"] = by_weight.reset_index().astype({"група ваги": str}).to_dict("records")

by_weight.rename(columns={"n": "каменів", "median_price": "медіана ціни, $",
                          "MAE": "MAE, $", "median_rel_err": "медіана відн. помилки, %"})

In [ ]:
#| label: worst
expensive = err[err["price"] > 15000]
big_stones = err[err["carat"] >= 1.5]

KEY["n_cap"] = len(expensive)
KEY["mae_cap"] = rnd(expensive["abs_err"].mean())
KEY["share_big_n"] = rnd(len(big_stones) / len(err) * 100)
KEY["share_big_err"] = rnd(big_stones["abs_err"].sum() / err["abs_err"].sum() * 100)

worst = err.sort_values("abs_err", ascending=False).head(6)
worst[["carat", "cut", "color", "clarity", "price", "pred", "abs_err"]].round(0)

**Де помилки найбільші.** У доларах помилка росте з вагою й ціною: камені від 1.5 ct становлять
`{python} KEY["share_big_n"]` % тесту, але дають `{python} KEY["share_big_err"]` % сумарної абсолютної
помилки. У відсотках картина інша: медіанна відносна помилка ≈ 5 % для каменів 0.5–2 ct, але
`{python} KEY["by_carat"][0]["median_rel_err"]` % для дрібних (до 0.5 ct) і
`{python} KEY["by_carat"][-1]["median_rel_err"]` % для каменів понад 2 ct. Розрив на графіку біля 1 500 $
походить з даних: у наборі майже немає цін між 1 455 і 1 545 $ (див. лабораторну №2).
Найбільші абсолютні помилки — у дорогих каменів поблизу верхньої межі цін набору (`{python} KEY["n_cap"]` каменів дорожчих за 15 000 $, MAE `{python} usd(KEY["mae_cap"])`):
там ціни обрізані, а інші характеристики (флуоресценція, пропорції, сертифікат, продавець) у дані не
потрапили.

In [ ]:
#| label: save-key
with open("results/key_numbers.json", "w", encoding="utf-8") as file:
    json.dump(KEY, file, ensure_ascii=False, indent=2, default=str)

# Обговорення й висновки

**Відповідь на дослідницьке питання.** Чотири характеристики з сертифіката прогнозують ціну нового
діаманта з того самого ринку дуже добре. Обрана модель на тесті помиляється в середньому на
`{python} usd(KEY["test"][1]["MAE, $"])`, тоді як константний прогноз — на
`{python} usd(KEY["test"][0]["MAE, $"])`. Типова (медіанна) відносна помилка становить
`{python} KEY["test"][1]["медіана |помилки| / ціна, %"]` %. Найкраща лінійна модель трохи гірша
(MAE `{python} usd(KEY["test"][-1]["MAE, $"])`), зате пояснює, як саме 4C пов'язані з ціною.

**H1 (вага).** Еластичність `{python} KEY["b1"]`, 95 % ДІ `{python} KEY["b1_lo"]`–`{python} KEY["b1_hi"]`.
Інтервал вузький і цілком лежить вище 1, тож за сталих інших C ціна зростає з вагою швидше, ніж
пропорційно. Дані **узгоджуються з H1**. Оцінка більша за нахил 1.68 з лабораторної №2, бо там вага
не була «очищена» від якості: великі камені частіше мають гіршу чистоту й колір. Застереження:
діагностика показала, що еластичність не стала (стрибки на круглих вагах), а гіпотеза походить з EDA,
тож результат розвідувальний.

**H2 (чистота).** Усі 7 коефіцієнтів чистоти додатні, з вузькими ДІ, і зростають від SI2 до IF
(`{python} "монотонно" if KEY["clar_monotone"] else "не зовсім монотонно"`). Надбавка IF над I1 становить
`{python} KEY["if_prem"]` % (95 % ДІ `{python} KEY["if_lo"]`–`{python} KEY["if_hi"]` %), що більше за
100 %. Дані **узгоджуються з H2** (так само розвідувально).

**Якість прогнозу на нових даних.** Різниця між MAE на перехресній валідації й на тесті мала, отже модель
не перенавчена. Однак тест не повністю незалежний: рішення про логарифми й набір предикторів я ухвалив
після EDA всього набору.

**Межі узагальнення й змішувальні чинники.**

- Дані — один ринок і один рік (2008), лише круглі камені, ціни до 18 823 $. На інші роки, валюти,
  форми каменів чи синтетичні діаманти результат не переноситься.
- Це **спостережні** дані, тому коефіцієнти описують **асоціації** в каталогах, а не причинний вплив.
  Змішувальні чинники, яких немає в даних: продавець і його націнка, лабораторія сертифікації (GIA чи інша),
  флуоресценція, симетрія й полірування, дата пропозиції.
- Ціни взято з каталогу (ціни пропозиції), а не з реальних угод.

**Продовження дослідження.** Зібрати свіжі дані з кількох онлайн-магазинів з датою, продавцем,
лабораторією сертифікації й флуоресценцією. Перевірити зафіксовані моделі (M5 для прогнозу, M4 для
інтерпретації) на них як на справді
незалежному тесті та оцінити, чи збереглися надбавки за «круглі» ваги через 15+ років.

# Бібліографія {.unnumbered}

::: {#refs}
:::